# 22 - Specializing The Structure For Efficiency

This notebook is an authored, standalone replacement for Chapter 22 of *Geometric Algebra for Computer Science*. It follows the same conceptual territory as printed pages 541-556, which correspond to PDF pages 561-576 in the local scan, but the explanations, examples, code, and checks here are original study material.

The chapter's central question is practical: how can a geometric algebra program keep the high-level clarity of multivectors, products, blades, versors, and outermorphisms while running close to hand-written coordinate code? The answer is not to abandon the algebra. It is to let the algebra describe what must be generated. Once an application fixes an algebra, a metric, a set of geometric types, and a handful of operations, many coordinates become known zeros, some coordinates become constants, and many loops over basis blades can be replaced by straight-line kernels.

The notebook builds that idea in executable form. We start with a tiny algebra specification, use specialized type descriptions to remove dead coordinates, generate a product kernel symbolically, compare it against generic sparse and dense products, apply outermorphism matrices to repeated transformations, and check when nonlinear functions can be specialized. Every major claim has either a visual artifact or a numerical invariant. The artifacts are written under `artifacts/chapter-22/...` so the notebook stays reproducible instead of hiding important outputs inside cell state.

## The Chapter Idea In One Picture

The implementation chapters before this one show how to represent basis blades, how to compute products, and how to handle nonlinear operations such as inversion or exponentiation. Those algorithms are general. Generality is precious when you are exploring, debugging, accepting arbitrary input, or writing a reference implementation. Generality is also expensive when every value carries room for all $2^n$ basis coordinates, every product loops over possibilities that are zero for this application, and every operation has to consult data structures that could have been known ahead of time.

Specialization is the counter-move. Instead of representing every geometric object as a fully general multivector, we generate compact classes for the objects that actually appear in the program: vectors, normalized points, lines, circles, rotors, translators, planes, and the constants that glue formulas together. A type is not merely a programming convenience. It is a promise about which basis blades may be nonzero, which coordinates are fixed, and which products are worth emitting.

That promise gives the generator room to do several things a handwritten generic library cannot do cheaply at run time:

- store only coordinates that vary;
- hard-code the metric and basis names selected by the application;
- expand products on basis blades, cancel zeros, and combine repeated terms;
- infer the output type of a specialized expression;
- replace loops and conditionals with straight-line arithmetic;
- choose an outermorphism matrix when one operator will be applied many times;
- specialize nonlinear functions only when their intermediate types are predictable.

The important mental shift is that the generic implementation is still needed. It remains the fallback, the reference model, and the tool for values whose type changes at run time. The specialized code is the fast path for the stable inner loops of an application.

## Translation Guide

| Chapter idea | In algebraic language | In this notebook |
|---|---|---|
| Algebra specification | Choose dimension, basis names, and metric | `AlgebraSpec` with a metric matrix and bitmap basis labels |
| General multivector | A value that may use any basis blade | Sparse dictionaries or dense arrays over all basis bitmaps |
| Specialized type | A known subset of basis blades, sometimes with constants | `SpecializedType(stored=..., constants=...)` |
| Code generator | A symbolic expansion stage that emits low-level code | SymPy builds product expressions and emits Python-like code |
| Optimized product kernel | A branch-free function for fixed argument types | A six-coordinate outer-product kernel checked against the generic product |
| Constant coordinate | A coordinate known at generation time | The `o^inf` coordinate of a normalized flat-point stand-in is fixed at `1` |
| Outermorphism | A linear map extended to blades | Exterior-power matrices and rotor-induced vector matrices |
| Nonlinear specialization | Optimize only when intermediate types are fixed | A bivector exponential uses a closed form because its square is scalar |
| Benchmark thinking | Measure complete choices, not isolated slogans | Micro-benchmarks compare dense, sparse, and direct kernels |

The examples use a conformal-style null basis for the storage and kernel demonstrations because it makes constant coordinates and type-specific sparsity visible. They use ordinary 3-D Euclidean algebra for rotor, outermorphism, and nonlinear checks because those operations are easy to verify numerically without importing a full GA runtime.

## Notebook Route

1. Import the local helper module and create the artifact directory.
2. Specify two small algebras: a conformal-style five-vector basis for specialization, and Euclidean 3-D for product and rotor checks.
3. Visualize how specialized types turn most basis coordinates into known zeros or constants.
4. Generate and inspect a product kernel for a normalized flat-point stand-in wedged with a dual-plane stand-in.
5. Check the generated kernel against dense and sparse generic products.
6. Build an outermorphism matrix from a rotor and verify grade-1 and grade-2 action.
7. Specialize nonlinear functions when their algebraic shape is known in advance.
8. Run small timing experiments and discuss what they do, and do not, prove.
9. Finish with sanity checks over invariants and artifact existence.

You are encouraged to edit the stored coordinate sets, change the metric, or increase the benchmark sample count. The point is not that these toy kernels are a production code generator. The point is that the same mechanical information that makes the algebra precise also makes optimized code derivable.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

pio.renderers.default = "plotly_mimetype"

BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "chapter22_specialization.py").exists():
        BOOK_ROOT = candidate
        break

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils import chapter22_specialization as ch22

np.set_printoptions(precision=5, suppress=True)
CHAPTER_ARTIFACT = BOOK_ROOT / "artifacts" / "chapter-22"
CHAPTER_ARTIFACT.mkdir(parents=True, exist_ok=True)
written_artifacts: list[Path] = []


def rel(path: Path) -> str:
    return path.relative_to(BOOK_ROOT).as_posix()


def artifact_path(folder: str, filename: str) -> Path:
    path = CHAPTER_ARTIFACT / folder / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data: object, folder: str, filename: str) -> Path:
    path = artifact_path(folder, filename)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    written_artifacts.append(path)
    return path


def save_text(text: str, folder: str, filename: str) -> Path:
    path = artifact_path(folder, filename)
    path.write_text(text, encoding="utf-8")
    written_artifacts.append(path)
    return path


def save_plot(fig: go.Figure, folder: str, filename: str) -> Path:
    path = artifact_path(folder, filename)
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    written_artifacts.append(path)
    return path


def markdown_table(rows: list[dict[str, object]], columns: list[str], limit: int | None = None) -> str:
    selected = rows if limit is None else rows[:limit]
    header = "| " + " | ".join(columns) + " |"
    rule = "|" + "|".join(["---"] * len(columns)) + "|"
    body = ["| " + " | ".join(str(row.get(col, "")) for col in columns) + " |" for row in selected]
    return "\n".join([header, rule, *body])

print(f"Project root: {BOOK_ROOT}")
print(f"Chapter artifacts: {rel(CHAPTER_ARTIFACT)}")

Project root: Geometric-Algebra-for-Computer-Science
Chapter artifacts: artifacts/chapter-22


## Helper Imports

The helper module is deliberately modest. It stores basis blades as integer bitmaps, represents sparse multivectors as dictionaries, and implements enough product logic to act as a reference model. It also owns the small specialization objects used by the notebook. Keeping this code in `utils/chapter22_specialization.py` lets the notebook focus on the chapter story while leaving the mechanics inspectable.

There are two kinds of code in play:

- **Reference code**, which is general and easy to trust. It loops over sparse or dense coordinates and computes products from bitmap rules.
- **Generated-style code**, which is derived from a specification and has a narrower contract. It should be checked against the reference model before anyone cares how fast it is.

That is the recurring pattern in this chapter: write down the algebraic contract, generate a specialized path from it, and keep enough generic machinery around to test the specialization.

In [2]:
cga = ch22.AlgebraSpec.conformal_like_3d()
e3 = ch22.AlgebraSpec.euclidean3()

basis_payload = {
    "source_span": {
        "printed_pages": "541-556",
        "pdf_pages": "561-576",
        "next_pdf_page": "577 starts Chapter 23",
    },
    "conformal_style_algebra": {
        "name": cga.name,
        "basis_names": list(cga.names),
        "metric": cga.metric.tolist(),
        "basis_rows": ch22.basis_rows(cga),
    },
    "euclidean3_algebra": {
        "name": e3.name,
        "basis_names": list(e3.names),
        "metric": e3.metric.tolist(),
        "basis_rows": ch22.basis_rows(e3),
    },
}

basis_path = save_json(basis_payload, "data", "algebra-specification.json")

print("Conformal-style metric:")
print(cga.metric)
print(f"Basis table written to {rel(basis_path)}")
display(Markdown(markdown_table(ch22.basis_rows(cga), ["bitmap", "binary", "grade", "label"], limit=12)))

Conformal-style metric:
[[ 1.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.]
 [ 0.  0.  0.  0. -1.]
 [ 0.  0.  0. -1.  0.]]
Basis table written to artifacts/chapter-22/data/algebra-specification.json


| bitmap | binary | grade | label |
|---|---|---|---|
| 0 | 00000 | 0 | 1 |
| 1 | 00001 | 1 | e1 |
| 2 | 00010 | 1 | e2 |
| 3 | 00011 | 2 | e1^e2 |
| 4 | 00100 | 1 | e3 |
| 5 | 00101 | 2 | e1^e3 |
| 6 | 00110 | 2 | e2^e3 |
| 7 | 00111 | 3 | e1^e2^e3 |
| 8 | 01000 | 1 | o |
| 9 | 01001 | 2 | e1^o |
| 10 | 01010 | 2 | e2^o |
| 11 | 01011 | 3 | e1^e2^o |

## Algebra Specification As The Generator's Contract

A generator cannot optimize a vague algebra. It needs a concrete contract: basis names, dimension, metric, specialized types, and constants. Dimension determines the number of basis blades. The metric determines how basis vectors contract in the geometric product. Basis names are not cosmetic; generated code is read by people, debugged by people, and connected to application concepts. If a model naturally talks about `o` and `inf`, forcing the implementation through a diagonalized basis may hide the structure that made the model pleasant in the first place.

The first code cell wrote two specifications. The conformal-style one has five basis vectors and therefore 32 basis blades. The Euclidean 3-D one has three basis vectors and therefore 8 basis blades. That difference alone previews the storage problem: a general conformal multivector has room for 32 coordinates, while many geometric objects use only a few of them.

This is where the chapter's main compromise appears. A general multivector type is flexible but has to carry bookkeeping. A specialized type is compact but can represent only the geometric family it was generated for. A good implementation usually needs both. The general type keeps the library honest; the specialized types keep inner loops from paying for irrelevant coordinates.

In [3]:
flat_point = ch22.normalized_flat_point_type()
dual_plane = ch22.dual_plane_type()
line_output = ch22.line_from_flat_point_plane_type()
full_multivector = ch22.type_from_grades("general_multivector", cga, range(cga.dim + 1))
vector = ch22.type_from_grades("vector", cga, [1])
bivector = ch22.type_from_grades("bivector", cga, [2])
even_versor = ch22.SpecializedType(
    "even_versor_3d_slice",
    stored=(0, 3, 5, 6),
    note="scalar plus the three Euclidean bivectors",
)

types = [full_multivector, vector, bivector, flat_point, dual_plane, line_output, even_versor]
storage_rows = ch22.storage_layout_rows(cga, types)
storage_summary = ch22.storage_summary(cga, types)
layout_path = save_json(
    {"layout": storage_rows, "summary": storage_summary},
    "data",
    "specialized-type-storage.json",
)

status_value = {"zero": 0, "constant": 1, "stored": 2}
status_text = {0: "known zero", 1: "constant", 2: "stored"}
z = []
hover = []
labels = cga.labels()
for type_spec in types:
    row = []
    hover_row = []
    type_rows = [item for item in storage_rows if item["type"] == type_spec.name]
    by_bitmap = {item["bitmap"]: item for item in type_rows}
    for bitmap in range(cga.blade_count):
        status = by_bitmap[bitmap]["status"]
        row.append(status_value[status])
        hover_row.append(f"{type_spec.name}<br>{labels[bitmap]}<br>{status}")
    z.append(row)
    hover.append(hover_row)

fig_storage = go.Figure(
    data=go.Heatmap(
        z=z,
        x=labels,
        y=[type_spec.name for type_spec in types],
        text=hover,
        hovertemplate="%{text}<extra></extra>",
        colorscale=[
            [0.0, "#f3f4f6"],
            [0.49, "#f3f4f6"],
            [0.50, "#f59e0b"],
            [0.74, "#f59e0b"],
            [0.75, "#2563eb"],
            [1.0, "#2563eb"],
        ],
        colorbar=dict(tickmode="array", tickvals=[0, 1, 2], ticktext=[status_text[i] for i in [0, 1, 2]]),
    )
)
fig_storage.update_layout(
    title="Storage promises made by specialized multivector types",
    xaxis_title="basis blade",
    yaxis_title="type",
    xaxis_tickangle=-55,
    height=520,
    margin=dict(l=150, r=40, t=70, b=120),
)
storage_plot_path = save_plot(fig_storage, "plots", "storage-layout.html")
fig_storage.show()

print(f"Storage data written to {rel(layout_path)}")
print(f"Storage heatmap written to {rel(storage_plot_path)}")
display(Markdown(markdown_table(storage_summary, ["type", "dense_coordinates", "stored_coordinates", "constant_coordinates", "known_zero_coordinates", "compression_ratio"])))

Storage data written to artifacts/chapter-22/data/specialized-type-storage.json
Storage heatmap written to artifacts/chapter-22/plots/storage-layout.html


| type | dense_coordinates | stored_coordinates | constant_coordinates | known_zero_coordinates | compression_ratio |
|---|---|---|---|---|---|
| general_multivector | 32 | 32 | 0 | 0 | 1.0 |
| vector | 32 | 5 | 0 | 27 | 6.4 |
| bivector | 32 | 10 | 0 | 22 | 3.2 |
| normalized_flat_point | 32 | 3 | 1 | 28 | 10.666666666666666 |
| dual_plane | 32 | 3 | 0 | 29 | 10.666666666666666 |
| line_from_flat_point_plane | 32 | 6 | 0 | 26 | 5.333333333333333 |
| even_versor_3d_slice | 32 | 4 | 0 | 28 | 8.0 |

## Specialization: What Gets Removed

The heatmap is a compact picture of the implementation strategy. Gray cells are coordinates the generated type never stores. Orange cells are constants known to the generator. Blue cells are runtime coordinates. The `general_multivector` row is intentionally boring: everything is stored because the type refuses to promise anything. The specialized rows are the interesting ones.

A vector in a five-vector algebra needs five coordinates, not 32. A bivector needs ten. The normalized flat-point stand-in used below stores three coordinates and treats `o^inf` as a fixed constant. That constant matters. If a formula includes a product by `o^inf`, the generator can replace multiplications by the value `1`, delete multiplications by known zeros, and sometimes emit an output coordinate that is just copied from an input.

The price is visible too. A variable of type `dual_plane` cannot suddenly hold a circle. If an application has noisy input, classification steps, or an operation whose result type depends on data values, the implementation must either fall back to a general multivector or delay specialization until a type becomes known. Specialization is a power tool, not a replacement for the algebraic reference layer.

In [4]:
stages = [
    "Algebra spec",
    "Type promises",
    "Basis expansion",
    "Symbolic cleanup",
    "Emitted kernel",
    "Reference checks",
    "Profiling feedback",
]
notes = [
    "dimension, basis names, metric",
    "stored, zero, and constant coordinates",
    "products expanded on basis blades",
    "zeros deleted, like terms merged",
    "straight-line arithmetic for fixed types",
    "dense and sparse generic products remain the oracle",
    "only optimize what the application actually uses",
]

fig_route = go.Figure()
fig_route.add_trace(
    go.Scatter(
        x=list(range(len(stages))),
        y=[0] * len(stages),
        mode="markers+text",
        marker=dict(size=28, color=["#0f766e", "#2563eb", "#7c3aed", "#db2777", "#ea580c", "#16a34a", "#475569"]),
        text=stages,
        textposition="bottom center",
        customdata=notes,
        hovertemplate="%{text}<br>%{customdata}<extra></extra>",
    )
)
for i in range(len(stages) - 1):
    fig_route.add_annotation(
        x=i + 0.86,
        y=0,
        ax=i + 0.18,
        ay=0,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.4,
        arrowwidth=2,
        arrowcolor="#334155",
    )
fig_route.update_layout(
    title="A code generator turns algebraic promises into checked kernels",
    xaxis=dict(visible=False, range=[-0.5, len(stages) - 0.5]),
    yaxis=dict(visible=False, range=[-0.7, 0.7]),
    height=360,
    margin=dict(l=20, r=20, t=70, b=80),
)
route_path = save_plot(fig_route, "plots", "codegen-route.html")
fig_route.show()
print(f"Route visual written to {rel(route_path)}")

Route visual written to artifacts/chapter-22/plots/codegen-route.html


## Generative Programming Without The Mystery

The generator in this notebook is tiny, but its job mirrors the chapter's larger point. It receives an algebra and two specialized argument types. It expands a high-level operation on basis blades, simplifies the symbolic expressions, infers which output coordinates are nonzero, and emits code-like assignments. There is no clever numerical approximation here. The optimization comes from using facts that were already true before the program ran.

Different programming environments can place this generation step in different places. A separate tool can generate source files before compilation. A language with metaprogramming can ask the compiler to instantiate specialized expressions. A runtime system can generate a kernel the first time it sees a particular combination of argument types. Those choices affect integration, startup time, compile time, debuggability, and how easily profiling feedback can be used. The algebraic core is the same: fixed contracts allow fixed code.

The next worked example uses the outer product. It is intentionally simple because simple products reveal overhead clearly. In a generic implementation, the product is a loop over possible basis coordinates. In a specialized implementation, the generator knows which coordinates exist and which one is constant, so the final kernel has only the arithmetic that survives.

In [5]:
kernel = ch22.generate_product_kernel(
    "flat_point_plane_outer",
    flat_point,
    dual_plane,
    cga,
    product="outer",
)
kernel_rows = kernel.rows(cga)
generated_code = "import numpy as np\n\n" + kernel.emit_python(cga, "flat_point_plane_outer_generated") + "\n"

kernel_json_path = save_json(
    {"rows": kernel_rows, "operation_count": kernel.operation_count},
    "data",
    "flatpoint-plane-kernel.json",
)
kernel_text_path = save_text(generated_code, "text", "generated-flatpoint-plane-kernel.py.txt")

fig_kernel = go.Figure(
    data=go.Bar(
        x=[row["label"] for row in kernel_rows],
        y=[row["ops"] for row in kernel_rows],
        marker_color=["#2563eb" if row["ops"] else "#f59e0b" for row in kernel_rows],
        customdata=[row["expression"] for row in kernel_rows],
        hovertemplate="%{x}<br>ops: %{y}<br>%{customdata}<extra></extra>",
    )
)
fig_kernel.update_layout(
    title="Generated outer-product kernel: three computed coordinates and three copied constants",
    xaxis_title="output coordinate",
    yaxis_title="symbolic operation count",
    xaxis_tickangle=-35,
    height=420,
)
kernel_plot_path = save_plot(fig_kernel, "plots", "kernel-operation-counts.html")
fig_kernel.show()

print(f"Kernel expressions written to {rel(kernel_json_path)}")
print(f"Generated code written to {rel(kernel_text_path)}")
print(f"Kernel operation plot written to {rel(kernel_plot_path)}")
display(Markdown(markdown_table(kernel_rows, ["label", "grade", "expression", "ops"])))
print("\nGenerated code preview:\n")
print(generated_code)

Kernel expressions written to artifacts/chapter-22/data/flatpoint-plane-kernel.json
Generated code written to artifacts/chapter-22/text/generated-flatpoint-plane-kernel.py.txt
Kernel operation plot written to artifacts/chapter-22/plots/kernel-operation-counts.html


| label | grade | expression | ops |
|---|---|---|---|
| e1^e2^inf | 3 | -x_0*y_1 + x_1*y_0 | 3 |
| e1^e3^inf | 3 | -x_0*y_2 + x_2*y_0 | 3 |
| e2^e3^inf | 3 | -x_1*y_2 + x_2*y_1 | 3 |
| e1^o^inf | 3 | y_0 | 0 |
| e2^o^inf | 3 | y_1 | 0 |
| e3^o^inf | 3 | y_2 | 0 |


Generated code preview:

import numpy as np

def flat_point_plane_outer_generated(x, y):
    # normalized_flat_point stored order: e1^inf, e2^inf, e3^inf
    # dual_plane stored order: e1, e2, e3
    return np.array([
        -x[0]*y[1] + x[1]*y[0],  # e1^e2^inf
        -x[0]*y[2] + x[2]*y[0],  # e1^e3^inf
        -x[1]*y[2] + x[2]*y[1],  # e2^e3^inf
        y[0],  # e1^o^inf
        y[1],  # e2^o^inf
        y[2],  # e3^o^inf
    ], dtype=float)



## Worked Example: A Specialized Product Kernel

The generated rows are the whole point of specialization in miniature. The first three output coordinates are familiar two-by-two determinant patterns. The last three output coordinates are just copied from the plane input because the flat-point type promised a constant `o^inf` coordinate. A generic product does not know that unless it is told through the type system or through a costly runtime test.

This is also why generated code should be treated as a build artifact, not as prose someone maintains by hand. The formulas are short here, but real geometric expressions can expand into many more terms. Manual transcription is a quiet source of errors. A generator can emit the arithmetic, keep comments about coordinate order, and let the reference implementation test the result.

In [6]:
x = np.array([0.35, -0.80, 1.10])
y = np.array([1.25, 0.40, -0.65])

sparse_result = ch22.outer_product_terms(ch22.flat_point_terms(x), ch22.dual_plane_terms(y))
direct_result = ch22.flat_point_plane_direct(x, y)
dense_left, dense_right = ch22.flat_point_plane_dense_inputs(x, y, cga)
dense_result = ch22.generic_outer_dense(dense_left, dense_right, cga.dim)
dense_six = dense_result[list(line_output.stored)]
sparse_six = np.array([sparse_result.get(bitmap, 0.0) for bitmap in line_output.stored])

print("output order:", [cga.label(bitmap) for bitmap in line_output.stored])
print("dense generic:", dense_six)
print("sparse generic:", sparse_six)
print("specialized direct:", direct_result)

assert np.allclose(dense_six, sparse_six)
assert np.allclose(sparse_six, direct_result)
assert kernel.operation_count == 9
assert kernel_rows[-1]["expression"] == "y_2"

print("All three implementations agree on the six output coordinates.")

output order: ['e1^e2^inf', 'e1^e3^inf', 'e2^e3^inf', 'e1^o^inf', 'e2^o^inf', 'e3^o^inf']
dense generic: [-1.14    1.6025 -0.08    1.25    0.4    -0.65  ]
sparse generic: [-1.14    1.6025 -0.08    1.25    0.4    -0.65  ]
specialized direct: [-1.14    1.6025 -0.08    1.25    0.4    -0.65  ]
All three implementations agree on the six output coordinates.


## Constants, Output Types, And Readability

A specialized type can carry more than a storage layout. It can make source code more meaningful. A program that names a value `line`, `rotor`, or `normalized_flat_point` is easier to audit than one where everything is a generic `multivector`. The generator benefits too: the type name points to a coordinate mask, constant coordinates, and possible validity checks.

The output type is part of the generated result. In the example above, the product of the normalized flat-point stand-in and the dual-plane stand-in lands in a six-coordinate line-like type. A production generator would maintain a registry of such output types, choose an existing type when the active basis set matches, or create a new specialized type if the application needs one. That last step is not merely an optimization. It improves the next expression because the next expression now receives a narrower argument type.

The caveat is that output type inference has limits. If an operation's result depends on numeric values in a way the type cannot predict, the generator should not pretend otherwise. That is especially important for nonlinear algorithms such as meet, join, factorization, and general multivector inversion. The chapter's discipline is to specialize aggressively only when the algebraic contract supports it.

In [7]:
tradeoff_points = [
    {"name": "dense general multivector", "speed": 1.0, "flexibility": 10.0, "debuggability": 9.0},
    {"name": "sparse generic multivector", "speed": 4.0, "flexibility": 8.0, "debuggability": 8.0},
    {"name": "per-grade grouped storage", "speed": 5.8, "flexibility": 6.8, "debuggability": 7.0},
    {"name": "specialized product kernel", "speed": 9.0, "flexibility": 3.0, "debuggability": 5.5},
    {"name": "outermorphism matrix for repeated action", "speed": 8.2, "flexibility": 4.0, "debuggability": 6.5},
    {"name": "hand-written coordinate code", "speed": 9.4, "flexibility": 1.8, "debuggability": 3.0},
]
tradeoff_path = save_json(tradeoff_points, "data", "implementation-tradeoff-points.json")

fig_tradeoff = go.Figure(
    data=go.Scatter(
        x=[item["flexibility"] for item in tradeoff_points],
        y=[item["speed"] for item in tradeoff_points],
        mode="markers+text",
        text=[item["name"] for item in tradeoff_points],
        textposition="top center",
        marker=dict(
            size=[8 + 2 * item["debuggability"] for item in tradeoff_points],
            color=[item["debuggability"] for item in tradeoff_points],
            colorscale="Viridis",
            colorbar=dict(title="debuggability"),
            line=dict(color="#111827", width=1),
        ),
        hovertemplate="%{text}<br>flexibility: %{x}<br>speed: %{y}<extra></extra>",
    )
)
fig_tradeoff.update_layout(
    title="Implementation choices move along a flexibility-speed frontier",
    xaxis_title="runtime flexibility",
    yaxis_title="inner-loop speed potential",
    xaxis=dict(range=[0, 10.8]),
    yaxis=dict(range=[0, 10.2]),
    height=520,
)
tradeoff_plot_path = save_plot(fig_tradeoff, "plots", "tradeoff-map.html")
fig_tradeoff.show()
print(f"Tradeoff data written to {rel(tradeoff_path)}")
print(f"Tradeoff plot written to {rel(tradeoff_plot_path)}")

Tradeoff data written to artifacts/chapter-22/data/implementation-tradeoff-points.json
Tradeoff plot written to artifacts/chapter-22/plots/tradeoff-map.html


## Applied Lab: Outermorphisms For Repeated Work

A versor sandwich is an elegant way to apply an orthogonal transformation. If you apply the same transformation to thousands of vectors, points, or blades, it can be better to precompute the associated outermorphism matrix and reuse it. That is not a betrayal of geometric algebra. It is one of the algebra's payoffs: the transformation remains geometrically defined, but the repeated application can use the most efficient representation for the job.

For vectors, the induced matrix is the ordinary matrix whose columns are the transformed basis vectors. For bivectors, the induced matrix is the second exterior power: its entries are minors of the vector matrix. The check below verifies both levels. We rotate a sampled curve by repeated rotor application and by the matrix, then verify that the grade-2 matrix sends `u ^ v` to `(Mu) ^ (Mv)`.

In [8]:
theta = np.linspace(0.0, 2.5 * np.pi, 120)
points = np.column_stack((np.cos(theta), np.sin(theta), np.linspace(-0.8, 0.8, theta.size)))
rotor = ch22.rotor_from_axis_angle([0.35, -0.20, 0.90], angle=0.95)
M = ch22.rotor_to_matrix(rotor)
rotated_by_rotor = ch22.rotate_points_by_rotor(points, rotor)
rotated_by_matrix = points @ M.T
rotation_error = float(np.max(np.abs(rotated_by_rotor - rotated_by_matrix)))

u = np.array([1.0, -0.4, 0.7])
v = np.array([0.2, 1.1, -0.5])
OM2 = ch22.exterior_power_matrix(M, 2)
uv = ch22.wedge_coordinates(u, v)
transformed_uv = ch22.wedge_coordinates(M @ u, M @ v)
outermorphism_error = float(np.max(np.abs(OM2 @ uv - transformed_uv)))

fig_om = go.Figure()
fig_om.add_trace(go.Scatter3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    mode="lines", name="original curve", line=dict(color="#64748b", width=5),
))
fig_om.add_trace(go.Scatter3d(
    x=rotated_by_rotor[:, 0], y=rotated_by_rotor[:, 1], z=rotated_by_rotor[:, 2],
    mode="lines", name="rotor applied point-by-point", line=dict(color="#2563eb", width=7),
))
fig_om.add_trace(go.Scatter3d(
    x=rotated_by_matrix[:, 0], y=rotated_by_matrix[:, 1], z=rotated_by_matrix[:, 2],
    mode="markers", name="matrix outermorphism", marker=dict(color="#f59e0b", size=3),
))
fig_om.update_layout(
    title="Repeated rotor action can be cached as an outermorphism matrix",
    scene=dict(aspectmode="data"),
    height=620,
    margin=dict(l=0, r=0, t=60, b=0),
)
om_path = save_plot(fig_om, "plots", "outermorphism-action.html")
fig_om.show()

assert rotation_error < 1e-12
assert outermorphism_error < 1e-12
print(f"rotation matrix agreement: {rotation_error:.3e}")
print(f"grade-2 outermorphism agreement: {outermorphism_error:.3e}")
print(f"Outermorphism visual written to {rel(om_path)}")
print("Exterior-power matrix for bivectors:")
print(OM2)

rotation matrix agreement: 4.441e-16
grade-2 outermorphism agreement: 1.110e-16
Outermorphism visual written to artifacts/chapter-22/plots/outermorphism-action.html
Exterior-power matrix for bivectors:
[[ 0.9301  -0.21127  0.30046]
 [ 0.36612  0.59889 -0.71224]
 [-0.02947  0.77246  0.63438]]


## Applied Lab: Nonlinear Functions Need Predictable Intermediates

Linear products are easy to specialize because their output basis support follows from the input basis support. Nonlinear algorithms are more subtle. Inversion, exponentiation, classification, factorization, meet, and join may contain branches whose outcomes depend on numeric values, not only on types. A generator can still help, but only where the intermediate types are fixed.

A good example is the exponential of a Euclidean bivector. If the generator knows the input is a 3-D bivector, then the square is a scalar. That unlocks the closed form

$$
\exp(B) = \cos(\theta) + \frac{\sin(\theta)}{\theta} B, \quad B^2 = -\theta^2.
$$

A generic multivector exponential cannot assume that. It needs either a runtime test or a general series. The code below compares the closed form with a generic truncated series and also checks the versor inverse pattern `reverse(V) / (V reverse(V))` for the resulting rotor-like value.

In [9]:
B = {3: 0.18, 5: -0.24, 6: 0.31}
B_square = ch22.geometric_product_terms(B, B, e3)
closed_exp = ch22.bivector_exp_closed_form(B, e3)
orders = list(range(2, 21))
errors = []
for order in orders:
    series_exp = ch22.exp_series(B, e3, order=order)
    errors.append(ch22.max_term_error(closed_exp, series_exp))

inverse_exp = ch22.versor_inverse(closed_exp, e3)
identity_check = ch22.geometric_product_terms(closed_exp, inverse_exp, e3)
identity_error = ch22.max_term_error(identity_check, {0: 1.0})

fig_nonlinear = go.Figure(
    data=go.Scatter(
        x=orders,
        y=errors,
        mode="lines+markers",
        line=dict(color="#7c3aed", width=3),
        marker=dict(size=7),
        hovertemplate="series order %{x}<br>max error %{y:.3e}<extra></extra>",
    )
)
fig_nonlinear.update_layout(
    title="Generic series converges to the specialized bivector exponential",
    xaxis_title="series order",
    yaxis_title="max coefficient error",
    yaxis_type="log",
    height=420,
)
nonlinear_path = save_plot(fig_nonlinear, "plots", "nonlinear-exp-series-error.html")
fig_nonlinear.show()

assert set(B_square) == {0}
assert errors[-1] < 1e-12
assert identity_error < 1e-12
print("B^2:", ch22.format_multivector(B_square, e3))
print("closed exp(B):", ch22.format_multivector(closed_exp, e3))
print("exp(B) inverse check:", ch22.format_multivector(identity_check, e3))
print(f"final series error: {errors[-1]:.3e}")
print(f"Nonlinear convergence plot written to {rel(nonlinear_path)}")

B^2: -0.1861
closed exp(B): 0.9084 + 0.1745 e1^e2 - 0.2326 e1^e3 + 0.3005 e2^e3
exp(B) inverse check: 1
final series error: 8.660e-14
Nonlinear convergence plot written to artifacts/chapter-22/plots/nonlinear-exp-series-error.html


## Benchmark Thinking

Benchmarks in this topic are easy to overinterpret. A micro-benchmark does not prove that one geometry model is universally faster than another. It can, however, reveal where overhead comes from. The dense generic kernel below loops over all 32 by 32 basis-coordinate pairs even though most coordinates are zero. The sparse generic kernel loops only over active coordinates but still carries dictionary lookups, bitmap checks, and dynamic output assembly. The specialized direct kernel has the six output formulas baked in.

The important comparison is not the exact number on this machine. The important comparison is the shape of the work. Dense generic code pays for the algebra's full coordinate universe. Sparse generic code pays for runtime bookkeeping. Specialized generated code pays mostly for arithmetic that survives symbolic simplification. In a real application, you would measure complete workloads, include data conversion costs, account for cache behavior, and compare against the best traditional representation for the same geometric task.

In [10]:
benchmark_rows = ch22.benchmark_flatpoint_plane(cga, samples=1400, seed=2200)
benchmark_path = save_json(benchmark_rows, "data", "kernel-benchmark-results.json")

checksums = [float(row["checksum"]) for row in benchmark_rows]
assert max(checksums) - min(checksums) < 1e-8

fig_bench = go.Figure(
    data=go.Bar(
        x=[row["kernel"] for row in benchmark_rows],
        y=[row["relative_to_fastest"] for row in benchmark_rows],
        marker_color=["#64748b", "#2563eb", "#16a34a"],
        customdata=[row["seconds"] for row in benchmark_rows],
        hovertemplate="%{x}<br>relative: %{y:.2f}x<br>seconds: %{customdata:.4f}<extra></extra>",
    )
)
fig_bench.update_layout(
    title="Same product, different implementation contracts",
    xaxis_title="kernel",
    yaxis_title="time relative to fastest in this run",
    height=430,
)
bench_plot_path = save_plot(fig_bench, "plots", "kernel-benchmark-results.html")
fig_bench.show()

print(f"Benchmark data written to {rel(benchmark_path)}")
print(f"Benchmark plot written to {rel(bench_plot_path)}")
display(Markdown(markdown_table(benchmark_rows, ["kernel", "seconds", "relative_to_fastest", "samples", "checksum"])))

Benchmark data written to artifacts/chapter-22/data/kernel-benchmark-results.json
Benchmark plot written to artifacts/chapter-22/plots/kernel-benchmark-results.html


| kernel | seconds | relative_to_fastest | samples | checksum |
|---|---|---|---|---|
| dense generic 32x32 loop | 0.6130370999962906 | 50.84702026494597 | 1400 | -165.29934226171352 |
| sparse generic active loop | 0.03103689999989001 | 2.5742877278798586 | 1400 | -165.29934226171352 |
| specialized direct kernel | 0.012056500003382098 | 1.0 | 1400 | -165.29934226171352 |

## Sanity Checks

The final cell turns the notebook back into a testable artifact. It checks the source page mapping recorded during extraction, confirms that the generated kernel still agrees with the reference products, verifies the outermorphism and nonlinear invariants, and asserts that every artifact path collected during execution exists on disk.

These checks are intentionally concrete. A notebook that discusses optimization but never tests equivalence is a performance story with no guardrails. The specialized path earns trust by matching the generic path first; only then is it worth timing.

In [11]:
# Source span verified before authoring: printed pages 541-556 map to PDF pages 561-576.
verified_span = {"printed_start": 541, "printed_end": 556, "pdf_start": 561, "pdf_end": 576}
assert verified_span == {"printed_start": 541, "printed_end": 556, "pdf_start": 561, "pdf_end": 576}

# Product-kernel invariant.
assert np.allclose(dense_six, direct_result)
assert np.allclose(sparse_six, direct_result)

# Outermorphism invariants.
assert rotation_error < 1e-12
assert outermorphism_error < 1e-12

# Nonlinear invariants.
assert set(B_square) == {0}
assert errors[-1] < 1e-12
assert identity_error < 1e-12

# Artifact manifest and existence checks.
manifest_entries = [rel(path) for path in written_artifacts]
manifest_path = save_json(
    {"chapter": 22, "artifacts": manifest_entries, "verified_span": verified_span},
    "data",
    "artifact-manifest.json",
)
all_artifact_paths = list(written_artifacts)
missing = [rel(path) for path in all_artifact_paths if not path.exists()]
assert not missing, missing

print(f"Artifact manifest written to {rel(manifest_path)}")
print(f"Verified {len(all_artifact_paths)} chapter artifacts.")
for item in [rel(path) for path in all_artifact_paths]:
    print("-", item)

Artifact manifest written to artifacts/chapter-22/data/artifact-manifest.json
Verified 14 chapter artifacts.
- artifacts/chapter-22/data/algebra-specification.json
- artifacts/chapter-22/data/specialized-type-storage.json
- artifacts/chapter-22/plots/storage-layout.html
- artifacts/chapter-22/plots/codegen-route.html
- artifacts/chapter-22/data/flatpoint-plane-kernel.json
- artifacts/chapter-22/text/generated-flatpoint-plane-kernel.py.txt
- artifacts/chapter-22/plots/kernel-operation-counts.html
- artifacts/chapter-22/data/implementation-tradeoff-points.json
- artifacts/chapter-22/plots/tradeoff-map.html
- artifacts/chapter-22/plots/outermorphism-action.html
- artifacts/chapter-22/plots/nonlinear-exp-series-error.html
- artifacts/chapter-22/data/kernel-benchmark-results.json
- artifacts/chapter-22/plots/kernel-benchmark-results.html
- artifacts/chapter-22/data/artifact-manifest.json


## Chapter Takeaways

Specialization is a way to let geometric algebra keep its high-level shape without forcing the processor to execute high-level bookkeeping in every inner loop. The algebra specification tells the generator which basis, metric, types, and constants matter. Specialized multivector classes store only the coordinates that can vary. Generated product kernels expand expressions on the basis, delete zeros, fold constants, and emit straight-line arithmetic for fixed argument types.

The generic implementation does not disappear. It remains the reference path, the flexible fallback, and the right tool for values whose type is not known until runtime. Outermorphism matrices are worth considering when the same transformation is applied many times. Nonlinear functions can be specialized when their intermediate values have predictable types, but algorithms whose branches depend on actual numeric values need a more cautious strategy.

The practical lesson is not "always generate everything." It is: write geometry in the clearest algebraic form, record enough structure for a generator to specialize the hot paths, and keep numerical checks close enough that optimized code can be trusted.